# ⚽ SoccerStats en Google Colab

Análisis de un partido grabado con **cámara fija**: detección y tracking de jugadores, separación de equipos, minimapa 2D y estadísticas (distancia, velocidad, posesión, mapas de calor).

**Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)`.

Pasos:
1. Preparar el entorno
2. Elegir el vídeo
3. Calibrar el campo (una sola vez por posición de cámara)
4. Analizar
5. Estadísticas

## 1. Preparar el entorno

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "⚠️ Sin GPU: activa una en Entorno de ejecución"

In [ ]:
# Google Drive: aquí guardaremos vídeos y resultados para no perderlos al cerrar Colab.
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/SoccerStats'
import os
os.makedirs(f'{DRIVE_DIR}/videos', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/resultados', exist_ok=True)

El repositorio es privado: hace falta un **token de GitHub** con permiso de lectura
(GitHub → Settings → Developer settings → Fine-grained tokens → acceso *Contents: Read* a `SoccerStats`).
Se pide sin mostrarlo y no se guarda en el notebook.

In [ ]:
from getpass import getpass
import os, subprocess

REPO = 'GViciano/SoccerStats'
if not os.path.exists('/content/SoccerStats'):
    token = getpass('Token de GitHub: ')
    subprocess.run(['git', 'clone', '-q', f'https://{token}@github.com/{REPO}.git', '/content/SoccerStats'], check=True)
    del token
else:
    !git -C /content/SoccerStats pull -q

%cd /content/SoccerStats
!pip install -q -r requirements.txt

### Modelo de detección

* **Recomendado:** modelo YOLO ya afinado para fútbol (proyecto `roboflow/sports`): distingue *balón, portero, jugador y árbitro*.
* **Alternativa:** modelo genérico COCO (`yolo11l.pt`): detecta personas y balón, pero no distingue árbitros ni porteros.

In [ ]:
MODELO = 'futbol'   # 'futbol' o 'coco'

os.makedirs('modelos', exist_ok=True)
if MODELO == 'futbol':
    MODEL_PATH = 'modelos/football-player-detection.pt'
    if not os.path.exists(MODEL_PATH):
        !gdown -q -O {MODEL_PATH} "https://drive.google.com/uc?id=17PXFNlx-jI7VjVo_vQnB1sONjRyvoB-q"
else:
    MODEL_PATH = 'yolo11l.pt'   # se descarga solo la primera vez
print('Modelo:', MODEL_PATH)

## 2. Elegir el vídeo

Sube tu vídeo a `Mi unidad/SoccerStats/videos/` y pon su nombre abajo.
Si todavía no tienes vídeo, deja `USAR_EJEMPLO = True` para probar con un clip público
(es de retransmisión de TV, no de cámara fija, así que la calibración solo valdrá para un instante).

In [ ]:
USAR_EJEMPLO = True
NOMBRE_VIDEO = 'partido.mp4'

if USAR_EJEMPLO:
    VIDEO = 'datos/ejemplo.mp4'
    os.makedirs('datos', exist_ok=True)
    if not os.path.exists(VIDEO):
        !gdown -q -O {VIDEO} "https://drive.google.com/uc?id=12TqauVZ9tLAv8kWxTTBFWtgt2hNQ4_ZF"
else:
    VIDEO = f'{DRIVE_DIR}/videos/{NOMBRE_VIDEO}'

import cv2
cap = cv2.VideoCapture(VIDEO)
fps, n = cap.get(cv2.CAP_PROP_FPS), int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f'{VIDEO}: {int(cap.get(3))}x{int(cap.get(4))}, {fps:.0f} fps, {n/fps/60:.1f} min')
cap.release()

## 3. Calibrar el campo

Relacionamos píxeles del vídeo con metros reales del campo. Solo hay que hacerlo **una vez por posición de la cámara**.

1. Mira la imagen con cuadrícula y la imagen de referencia con los nombres de los puntos.
2. Escribe las coordenadas (x, y) en píxeles de **al menos 4 puntos** (mejor 6–8, repartidos por todo el campo).
3. Ejecuta la verificación: las líneas amarillas deben caer encima de las líneas reales.

In [ ]:
!python scripts/calibrar.py frame "{VIDEO}" --segundo 2 --salida salidas/frame.jpg
!python scripts/calibrar.py puntos --salida salidas/referencia_campo.png

import matplotlib.pyplot as plt
for path in ['salidas/frame_cuadricula.jpg', 'salidas/referencia_campo.png']:
    plt.figure(figsize=(18, 10)); plt.imshow(cv2.imread(path)[..., ::-1]); plt.axis('off'); plt.show()

In [ ]:
import json

CALIBRACION = {
    'campo': {'tipo': 'futbol11', 'length': 105, 'width': 68},   # pon las medidas reales si las sabes
    'puntos': {
        # 'nombre_del_punto': [x_pixel, y_pixel],   <- cambia los [0, 0] por coordenadas reales
        # Los que dejes a [0, 0] se ignoran. Puedes borrar o añadir puntos (ver imagen de referencia).
        'medio_sup': [0, 0],
        'medio_inf': [0, 0],
        'circulo_sup': [0, 0],
        'circulo_inf': [0, 0],
        'area_der_sup': [0, 0],
        'area_der_inf': [0, 0],
    },
}

CALIB_PATH = f'{DRIVE_DIR}/calibracion.json'
with open(CALIB_PATH, 'w') as f:
    json.dump(CALIBRACION, f, indent=2)

if os.path.exists('salidas/verificacion.jpg'):
    os.remove('salidas/verificacion.jpg')
!python scripts/calibrar.py verificar "{VIDEO}" "{CALIB_PATH}" --segundo 2 --salida salidas/verificacion.jpg
if os.path.exists('salidas/verificacion.jpg'):
    plt.figure(figsize=(18, 10)); plt.imshow(cv2.imread('salidas/verificacion.jpg')[..., ::-1]); plt.axis('off'); plt.show()
else:
    print('❌ Calibración no válida: lee el mensaje de arriba, corrige los puntos y vuelve a ejecutar esta celda.')

## 4. Analizar

Primero una prueba corta (`MAX_FRAMES = 300`, ~10 s). Cuando todo se vea bien, pon `MAX_FRAMES = None` para el partido entero
(en una T4, un partido de 90 min tarda del orden de 1–2 h; con `STRIDE = 2` va el doble de rápido).

In [ ]:
MAX_FRAMES = 300
STRIDE = 1
SALIDA = f'{DRIVE_DIR}/resultados/' + os.path.splitext(os.path.basename(VIDEO))[0]

extra = f'--max-frames {MAX_FRAMES}' if MAX_FRAMES else ''
!python scripts/analizar.py "{VIDEO}" --calibracion "{CALIB_PATH}" --modelo {MODEL_PATH} \
    --dispositivo 0 --stride {STRIDE} --salida "{SALIDA}" {extra}

In [ ]:
# Ver el vídeo anotado en el notebook (se convierte a H.264 para el navegador).
from IPython.display import HTML
from base64 import b64encode

!ffmpeg -y -loglevel error -i "{SALIDA}/anotado.mp4" -vcodec libx264 -crf 28 -t 30 /content/preview.mp4
data = b64encode(open('/content/preview.mp4', 'rb').read()).decode()
HTML(f'<video width=960 controls><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>')

## 5. Estadísticas

In [ ]:
!python scripts/estadisticas.py "{SALIDA}/tracks.csv" --calibracion "{CALIB_PATH}"

fig, axes = plt.subplots(1, 3, figsize=(24, 6))
for ax, name in zip(axes, ['calor_equipo_0.png', 'calor_equipo_1.png', 'calor_balon.png']):
    ax.imshow(cv2.imread(f'{SALIDA}/{name}')[..., ::-1]); ax.set_title(name); ax.axis('off')
plt.show()

Todos los resultados quedan en `Mi unidad/SoccerStats/resultados/`:

| Fichero | Contenido |
|---|---|
| `anotado.mp4` | vídeo con jugadores, equipos, IDs, balón y minimapa |
| `tracks.csv` | posición de cada jugador y del balón en cada frame (píxeles y metros) |
| `resumen_jugadores.csv` | distancia, velocidad media y máxima, sprints por jugador |
| `calor_*.png` | mapas de calor por equipo y del balón |

> ⚠️ Los IDs del tracking pueden cortarse cuando un jugador queda tapado; un mismo jugador puede aparecer con varios IDs.